# NB11: rebuilding the Gazette company features on notices through 31 July 2026

NB10 built 52 company-level Gazette features from notices covering 30 June 2023 to 30 June 2026.
We have since extended the notice window by one month, so this notebook rebuilds those same
features on the longer record and then works out what actually changed.

The rebuild matters more than it might sound. Adding July notices does not just add July
companies. It also moves the reference date used by every recency feature, so a company that
had no July activity at all can still see its numbers change simply because the clock moved on.
Separating those two effects is most of the work here.

**What this notebook does, in order**

1. Load the merged notice file (291,045 notices, 2023-06-30 to 2026-07-31).
2. Map notice types to families and drop the ceremonial noise, exactly as NB10 did.
3. Keep only notices carrying a CompanyNumber, since that is the only join key we trust.
4. Rebuild all 52 features with the reference date moved to 31 July 2026.
5. Build a control version on the old notices with the new reference date, so the clock effect
   and the new-notice effect can be told apart.
6. Compare against the existing June feature file and report what moved.
7. Reconcile the watchlist against both the old Active-only universe and the widened one.

**Nothing existing is overwritten.** The June feature file stays exactly where it is until the
comparison has been read and accepted. Every output here goes to a new filename.

The feature logic is copied from NB10 rather than rewritten, so any difference in the output is
caused by the data and not by us quietly changing a definition.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

DATA = Path(r"C:\Users\visha\Lloyds_Github\data\processed")

# Inputs
NOTICES_MERGED = DATA / "nb10_gazette_notices_thru_2026-07.csv"   # 291,045 notices
FEATURES_JUNE  = DATA / "nb10_gazette_company_features.csv"       # the existing file, read only

# Outputs, all new names so nothing existing is touched
FEATURES_JULY   = DATA / "nb10_gazette_company_features_thru_2026-07.csv"
WATCHLIST_JULY  = DATA / "nb10_active_gazette_watchlist_thru_2026-07.csv"

# Two reference dates. The old one is needed to reproduce the June file's behaviour,
# the new one is what the rebuild runs on. Pinned, never Timestamp.now().
SNAPSHOT_JUNE = pd.Timestamp("2026-06-30")
SNAPSHOT_JULY = pd.Timestamp("2026-07-31")

notices_raw = pd.read_csv(NOTICES_MERGED, dtype=str)
notices_raw.columns = [c.strip() for c in notices_raw.columns]
notices_raw["notice_date"] = pd.to_datetime(notices_raw["notice_date"], errors="coerce")

print("notices loaded :", f"{len(notices_raw):,}")
print("date range     :", notices_raw.notice_date.min().date(), "->", notices_raw.notice_date.max().date())
print("reference dates:", SNAPSHOT_JUNE.date(), "(old) and", SNAPSHOT_JULY.date(), "(new)")
print("\nnotices in the July extension:",
      f"{int((notices_raw.notice_date > SNAPSHOT_JUNE).sum()):,}")

notices loaded : 291,045
date range     : 2023-06-30 -> 2026-07-31
reference dates: 2026-06-30 (old) and 2026-07-31 (new)

notices in the July extension: 6,815


## Step 1: notice types to families

The Gazette publishes dozens of notice types, and only some of them describe corporate distress.
The rest are honours lists, proclamations and Crown office notices that happen to share the feed.
NB10 grouped the relevant types into eleven families and dropped everything else as noise, so we
use exactly the same definitions here.

The distress ladder sits on top of the families. It is an ordinal scale from 1 to 5 covering the
normal progression of an insolvency, from a petition being presented through to the final meeting.
Branch events such as a petition being dismissed do not sit on the ladder, because they are not a
step further into distress. Those are captured as separate flags instead.

In [2]:
FAMILY_DEFS = {
    "petition": {
        "Petitions to Wind Up (Companies)",
        "Petitions to Wind Up (Partnerships)",
        "Service of Petition",
    },
    "winding_up_order": {
        "Winding-Up Orders",
        "Winding-Up Orders (Partnerships)",
    },
    "voluntary_liquidation": {
        "Resolutions for Winding-up",
        "Resolutions for Winding Up",
        "Appointment of Liquidators",
    },
    "administration": {
        "Appointment of Administrators",
        "Administration Orders",
        "Appointment of Administrative Receivers",
    },
    "creditor_process": {
        "Notices to Creditors",
        "Meetings of Creditors",
        "Deemed Consent",
        "Qualifying Decision Procedure",
        "Notices to Members",
    },
    "dividend": {
        "Notice of Intended Dividends",
        "Notice of Dividends",
    },
    "closing": {
        "Final Meetings",
        "Annual Liquidation Meetings",
    },
    "petition_dismissed":    {"Dismissal of Winding Up Petition"},
    "prohibited_name_reuse": {"Moratoria, Prohibited Names and Other: Re-use of a Prohibited Name"},
    "cross_border":          {"Court Petitions and Orders: Cross-border Insolvencies"},
    "other_insolvency":      {"Other Corporate Insolvency Notices"},
}
TYPE_TO_FAMILY = {t: fam for fam, types in FAMILY_DEFS.items() for t in types}

STAGE_MAP = {
    "petition": 1,
    "winding_up_order": 2, "administration": 2, "voluntary_liquidation": 2,
    "creditor_process": 3,
    "dividend": 4,
    "closing": 5,
}
OPEN_STAGE_FAMILIES = {"petition", "winding_up_order", "administration", "voluntary_liquidation"}

df = notices_raw.copy()
df["family"] = df["notice_type"].map(TYPE_TO_FAMILY).fillna("noise")

print("notices per family:")
print(df["family"].value_counts().to_string())

noise_n = int((df["family"] == "noise").sum())
print(f"\ndropped as non-corporate noise: {noise_n:,} rows ({noise_n / len(df) * 100:.1f}%)")
print("example noise types:",
      sorted(df.loc[df.family == "noise", "notice_type"].dropna().unique())[:10])

notices per family:
family
voluntary_liquidation    183390
creditor_process          50622
petition                  19343
noise                     10983
winding_up_order          10959
administration             5575
dividend                   4977
prohibited_name_reuse      3265
closing                     866
petition_dismissed          800
other_insolvency            245
cross_border                 20

dropped as non-corporate noise: 10,983 rows (3.8%)
example noise types: ['Appointment of Sheriffs', 'Bar to the Royal Victorian Medal', 'British Empire Medal', 'Chancellor of the Exchequer', 'Crown Office', 'Deputy Lieutenant Commissions', 'Duchy of Cornwall or Duchy of Lancaster', 'Honours and Awards', "King's Ambulance Service Medal", "King's Fire Service Medal"]


## Step 2: keep what we can actually key

A notice without a company number cannot be attributed to a specific company. Name matching is
not an option here, because company names collide constantly in this dataset and a wrong match
would put a distress signal against an innocent business.

So notices without a number are excluded from the feature table and reported as a limitation
rather than guessed at. This is the single biggest honest caveat in the Gazette work, and the
proportion is large, so it is worth printing every time rather than burying it.

In [3]:
ins = df[df["family"] != "noise"].copy()
ins["CompanyNumber"] = ins["CompanyNumber"].str.strip()
has_num = ins["CompanyNumber"].notna() & ~ins["CompanyNumber"].isin(["", "nan", "None"])

print("corporate insolvency notices:", f"{len(ins):,}")
print(f"  with CompanyNumber (kept): {int(has_num.sum()):,} ({has_num.mean() * 100:.1f}%)")
print(f"  name-only (excluded)     : {int((~has_num).sum()):,} ({(~has_num).mean() * 100:.1f}%)")
print("  distinct excluded names  :",
      f"{ins.loc[~has_num, 'company_name'].str.upper().str.strip().nunique():,}",
      "(upper bound on companies we cannot key)")

keyed = ins[has_num].copy()
print("\nunique companies to build features for:", f"{keyed['CompanyNumber'].nunique():,}")

corporate insolvency notices: 280,062
  with CompanyNumber (kept): 161,125 (57.5%)
  name-only (excluded)     : 118,937 (42.5%)
  distinct excluded names  : 85,626 (upper bound on companies we cannot key)

unique companies to build features for: 109,333


## Step 3: the feature builder

This function is lifted from NB10 unchanged. It takes keyed notices plus a reference date and
returns one row per company with all 52 features.

It is a pure function, which is what makes the rest of this notebook possible. Because the
reference date is an argument rather than a global, we can run it twice on different inputs and
attribute every difference to exactly one cause.

In [4]:
def build_gazette_company_features(notices, snapshot_date):
    'Keyed insolvency notices plus a reference date, in. One row per company, out.'
    d = notices.copy()
    d["stage"] = d["family"].map(STAGE_MAP)
    d["is_open_stage"] = d["family"].isin(OPEN_STAGE_FAMILIES).astype(int)
    d["postcode"] = d["postcode"].fillna("").str.strip()
    d["postcode_present"] = (d["postcode"] != "").astype(int)

    fams = ["petition", "winding_up_order", "voluntary_liquidation", "administration",
            "creditor_process", "dividend", "closing", "petition_dismissed",
            "prohibited_name_reuse", "cross_border", "other_insolvency"]
    for f in fams:
        d[f"is_{f}"] = (d["family"] == f).astype(int)
    d["is_liquidation"] = ((d.family == "voluntary_liquidation") |
                           (d.family == "winding_up_order")).astype(int)
    d["is_receiver"] = (d["notice_type"] == "Appointment of Administrative Receivers").astype(int)
    d["is_partnership"] = d["notice_type"].str.contains("Partnership", na=False).astype(int)
    d["in_12m"] = (d.notice_date >= snapshot_date - pd.Timedelta(days=365)).astype(int)
    d["in_24m"] = (d.notice_date >= snapshot_date - pd.Timedelta(days=730)).astype(int)

    g = d.groupby("CompanyNumber", sort=False)
    feat = g.agg(
        company_name=("company_name", "first"),
        gaz_notice_count_total=("notice_type", "count"),
        gaz_distinct_notice_type_count=("notice_type", "nunique"),
        gaz_winding_up_petition_count=("is_petition", "sum"),
        gaz_winding_up_order_count=("is_winding_up_order", "sum"),
        gaz_voluntary_liquidation_count=("is_voluntary_liquidation", "sum"),
        gaz_liquidation_notice_count=("is_liquidation", "sum"),
        gaz_administration_count=("is_administration", "sum"),
        gaz_creditors_process_count=("is_creditor_process", "sum"),
        gaz_dividend_notice_count=("is_dividend", "sum"),
        gaz_closing_notice_count=("is_closing", "sum"),
        gaz_first_notice_date=("notice_date", "min"),
        gaz_latest_notice_date=("notice_date", "max"),
        gaz_notice_count_12m=("in_12m", "sum"),
        gaz_notice_count_24m=("in_24m", "sum"),
        gaz_max_distress_stage=("stage", "max"),
        _min_stage=("stage", "min"),
        gaz_has_prohibited_name_reuse=("is_prohibited_name_reuse", "max"),
        gaz_has_petition_dismissed=("is_petition_dismissed", "max"),
        gaz_has_cross_border_insolvency=("is_cross_border", "max"),
        gaz_has_other_insolvency=("is_other_insolvency", "max"),
        gaz_receiver_appointed_flag=("is_receiver", "max"),
        gaz_partnership_insolvency_flag=("is_partnership", "max"),
        gaz_postcode_present_flag=("postcode_present", "max"),
    )

    feat["gaz_has_any_notice"] = 1
    feat["gaz_has_winding_up_petition"] = (feat.gaz_winding_up_petition_count > 0).astype(int)
    feat["gaz_has_winding_up_order"] = (feat.gaz_winding_up_order_count > 0).astype(int)
    feat["gaz_has_liquidation"] = (feat.gaz_liquidation_notice_count > 0).astype(int)
    feat["gaz_has_administration"] = (feat.gaz_administration_count > 0).astype(int)
    feat["gaz_has_creditors_process"] = (feat.gaz_creditors_process_count > 0).astype(int)
    feat["gaz_has_dividend_notice"] = (feat.gaz_dividend_notice_count > 0).astype(int)
    feat["gaz_has_closing_notice"] = (feat.gaz_closing_notice_count > 0).astype(int)

    feat["gaz_compulsory_liquidation_flag"] = feat.gaz_has_winding_up_order
    feat["gaz_voluntary_liquidation_flag"] = ((feat.gaz_voluntary_liquidation_count > 0) &
                                              (feat.gaz_winding_up_order_count == 0)).astype(int)
    feat["gaz_court_involved_flag"] = ((feat.gaz_winding_up_petition_count > 0) |
                                       (feat.gaz_winding_up_order_count > 0)).astype(int)
    feat["gaz_max_distress_stage"] = feat.gaz_max_distress_stage.fillna(0).astype(int)

    def _tier(s):
        return ("none" if s == 0 else "early_warning" if s == 1
                else "formal_insolvency" if s == 2 else "terminal")
    feat["gaz_severity_tier"] = feat.gaz_max_distress_stage.map(_tier)

    feat["gaz_days_since_latest_notice"] = (snapshot_date - feat.gaz_latest_notice_date).dt.days
    feat["gaz_days_since_first_notice"] = (snapshot_date - feat.gaz_first_notice_date).dt.days
    feat["gaz_notice_span_days"] = (feat.gaz_latest_notice_date - feat.gaz_first_notice_date).dt.days
    feat["gaz_recent_notice_90d_flag"] = (feat.gaz_days_since_latest_notice <= 90).astype(int)
    feat["gaz_recent_notice_365d_flag"] = (feat.gaz_days_since_latest_notice <= 365).astype(int)

    _mins = feat["_min_stage"].fillna(feat.gaz_max_distress_stage)
    feat["gaz_stage_progressed_flag"] = ((feat.gaz_max_distress_stage - _mins) >= 2).astype(int)

    latest = (d.sort_values("notice_date").groupby("CompanyNumber", sort=False).tail(1)
                .set_index("CompanyNumber"))
    stage_label = {1: "petition", 2: "formal_insolvency", 3: "creditor_process",
                   4: "dividend", 5: "closing"}
    feat["gaz_current_stage"] = latest["stage"].map(stage_label).reindex(feat.index).fillna("other")
    feat["gaz_active_case_flag"] = ((latest["is_open_stage"].reindex(feat.index) == 1) &
                                    (feat.gaz_days_since_latest_notice <= 365)).astype(int)
    feat["gaz_notice_postcode"] = latest["postcode"].reindex(feat.index).replace("", np.nan)
    feat["gaz_notice_postcode_area"] = (feat["gaz_notice_postcode"]
                                        .str.extract(r"^([A-Za-z]{1,2})", expand=False).str.upper())
    feat["gaz_latest_notice_url"] = latest["notice_url"].reindex(feat.index)

    pet_min = d[d.is_petition == 1].groupby("CompanyNumber").notice_date.min().reindex(feat.index)
    adm_min = d[d.is_administration == 1].groupby("CompanyNumber").notice_date.min().reindex(feat.index)
    liq_min = d[d.is_liquidation == 1].groupby("CompanyNumber").notice_date.min().reindex(feat.index)
    liq_max = d[d.is_liquidation == 1].groupby("CompanyNumber").notice_date.max().reindex(feat.index)
    feat["gaz_petition_to_liquidation_flag"] = (pet_min.notna() & liq_max.notna() &
                                                (liq_max >= pet_min)).astype(int)
    feat["gaz_administration_to_liquidation_flag"] = (adm_min.notna() & liq_max.notna() &
                                                      (liq_max >= adm_min)).astype(int)
    dpo = (liq_min - pet_min).dt.days
    feat["gaz_days_petition_to_order"] = dpo.where(dpo >= 0)

    def _burst(dates):
        ds = sorted(dates)
        return int(any((ds[i + 2] - ds[i]).days <= 90 for i in range(len(ds) - 2)))

    def _recurring(dates):
        ds = sorted(set(dates))
        return int(len(ds) >= 2 and (ds[-1] - ds[0]).days > 180)

    date_lists = d.dropna(subset=["notice_date"]).groupby("CompanyNumber").notice_date.apply(list)
    feat["gaz_notice_burst_flag"] = date_lists.map(_burst).reindex(feat.index).fillna(0).astype(int)
    feat["gaz_recurring_distress_flag"] = date_lists.map(_recurring).reindex(feat.index).fillna(0).astype(int)

    return feat.drop(columns=["_min_stage"]).reset_index()


print("feature builder defined")

feature builder defined


## Step 4: build the new table, and a control

Two runs.

**The rebuild** uses everything we have, all notices through 31 July, with the reference date at
31 July. This is the deliverable.

**The control** uses only the old notices, those on or before 30 June, but with the new reference
date of 31 July. It is not a deliverable. It exists so that we can attribute changes properly.
Comparing June against the control isolates the effect of the clock moving, and comparing the
control against the rebuild isolates the effect of the new notices. Without it, the two effects
are tangled together and any statement about what changed would be guesswork.

In [5]:
keyed_june = keyed[keyed["notice_date"] <= SNAPSHOT_JUNE].copy()

print("building the rebuild (all notices, reference 2026-07-31) ...")
feat_july = build_gazette_company_features(keyed, SNAPSHOT_JULY)

print("building the control (June notices, reference 2026-07-31) ...")
feat_control = build_gazette_company_features(keyed_june, SNAPSHOT_JULY)

print("\nrebuild shape:", feat_july.shape)
print("control shape:", feat_control.shape)
print("\ncompanies gained by adding July notices:",
      f"{len(feat_july) - len(feat_control):,}")
feat_july.head(3)

building the rebuild (all notices, reference 2026-07-31) ...
building the control (June notices, reference 2026-07-31) ...

rebuild shape: (109333, 52)
control shape: (106664, 52)

companies gained by adding July notices: 2,669


,CompanyNumber,company_name,gaz_notice_count_total,gaz_distinct_notice_type_count,gaz_winding_up_petition_count,gaz_winding_up_order_count,gaz_voluntary_liquidation_count,gaz_liquidation_notice_count,gaz_administration_count,gaz_creditors_process_count,gaz_dividend_notice_count,gaz_closing_notice_count,gaz_first_notice_date,gaz_latest_notice_date,gaz_notice_count_12m,gaz_notice_count_24m,gaz_max_distress_stage,gaz_has_prohibited_name_reuse,gaz_has_petition_dismissed,gaz_has_cross_border_insolvency,gaz_has_other_insolvency,gaz_receiver_appointed_flag,gaz_partnership_insolvency_flag,gaz_postcode_present_flag,gaz_has_any_notice,gaz_has_winding_up_petition,gaz_has_winding_up_order,gaz_has_liquidation,gaz_has_administration,gaz_has_creditors_process,gaz_has_dividend_notice,gaz_has_closing_notice,gaz_compulsory_liquidation_flag,gaz_voluntary_liquidation_flag,gaz_court_involved_flag,gaz_severity_tier,gaz_days_since_latest_notice,gaz_days_since_first_notice,gaz_notice_span_days,gaz_recent_notice_90d_flag,gaz_recent_notice_365d_flag,gaz_stage_progressed_flag,gaz_current_stage,gaz_active_case_flag,gaz_notice_postcode,gaz_notice_postcode_area,gaz_latest_notice_url,gaz_petition_to_liquidation_flag,gaz_administration_to_liquidation_flag,gaz_days_petition_to_order,gaz_notice_burst_flag,gaz_recurring_distress_flag
0,04414254,MEC (GB) LTD,3,3,0,0,2,2,0,1,0,0,2026-06-30,2026-07-08,3,3,3,0,0,0,0,0,0,1,1,0,0,1,0,1,0,0,0,1,0,terminal,23,31,8,1,1,0,formal_insolvency,1,YO25 6DA,YO,https://www.thegazette.co.uk/id/notice/5170567,0,0,NaN,1,0
1,12722787,EXCEED CORPORATE EVENTS LTD,3,3,0,0,2,2,0,1,0,0,2026-06-30,2026-07-13,3,3,3,0,0,0,0,0,0,1,1,0,0,1,0,1,0,0,0,1,0,terminal,18,31,13,1,1,0,formal_insolvency,1,GL54 2RL,GL,https://www.thegazette.co.uk/id/notice/5172503,0,0,NaN,1,0
2,OC318745,GREENSTONE CAPITAL LLP,2,2,0,0,2,2,0,0,0,0,2026-06-30,2026-06-30,2,2,2,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,1,0,formal_insolvency,31,31,0,1,1,0,formal_insolvency,1,EC4V 4BE,EC,https://www.thegazette.co.uk/id/notice/5165331,0,0,NaN,0,0



rebuild shape: (109333, 52)
control shape: (106664, 52)

companies gained by adding July notices: 2,669


,CompanyNumber,company_name,gaz_notice_count_total,gaz_distinct_notice_type_count,gaz_winding_up_petition_count,gaz_winding_up_order_count,gaz_voluntary_liquidation_count,gaz_liquidation_notice_count,gaz_administration_count,gaz_creditors_process_count,gaz_dividend_notice_count,gaz_closing_notice_count,gaz_first_notice_date,gaz_latest_notice_date,gaz_notice_count_12m,gaz_notice_count_24m,gaz_max_distress_stage,gaz_has_prohibited_name_reuse,gaz_has_petition_dismissed,gaz_has_cross_border_insolvency,gaz_has_other_insolvency,gaz_receiver_appointed_flag,gaz_partnership_insolvency_flag,gaz_postcode_present_flag,gaz_has_any_notice,gaz_has_winding_up_petition,gaz_has_winding_up_order,gaz_has_liquidation,gaz_has_administration,gaz_has_creditors_process,gaz_has_dividend_notice,gaz_has_closing_notice,gaz_compulsory_liquidation_flag,gaz_voluntary_liquidation_flag,gaz_court_involved_flag,gaz_severity_tier,gaz_days_since_latest_notice,gaz_days_since_first_notice,gaz_notice_span_days,gaz_recent_notice_90d_flag,gaz_recent_notice_365d_flag,gaz_stage_progressed_flag,gaz_current_stage,gaz_active_case_flag,gaz_notice_postcode,gaz_notice_postcode_area,gaz_latest_notice_url,gaz_petition_to_liquidation_flag,gaz_administration_to_liquidation_flag,gaz_days_petition_to_order,gaz_notice_burst_flag,gaz_recurring_distress_flag
0,04414254,MEC (GB) LTD,3,3,0,0,2,2,0,1,0,0,2026-06-30,2026-07-08,3,3,3,0,0,0,0,0,0,1,1,0,0,1,0,1,0,0,0,1,0,terminal,23,31,8,1,1,0,formal_insolvency,1,YO25 6DA,YO,https://www.thegazette.co.uk/id/notice/5170567,0,0,NaN,1,0
1,12722787,EXCEED CORPORATE EVENTS LTD,3,3,0,0,2,2,0,1,0,0,2026-06-30,2026-07-13,3,3,3,0,0,0,0,0,0,1,1,0,0,1,0,1,0,0,0,1,0,terminal,18,31,13,1,1,0,formal_insolvency,1,GL54 2RL,GL,https://www.thegazette.co.uk/id/notice/5172503,0,0,NaN,1,0
2,OC318745,GREENSTONE CAPITAL LLP,2,2,0,0,2,2,0,0,0,0,2026-06-30,2026-06-30,2,2,2,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,1,0,formal_insolvency,31,31,0,1,1,0,formal_insolvency,1,EC4V 4BE,EC,https://www.thegazette.co.uk/id/notice/5165331,0,0,NaN,0,0


## Step 5: validation

Before comparing anything, the new table has to stand up on its own. The checks that matter are
that every company appears exactly once, and that the per-company notice counts add back up to
the number of notices we fed in. If that reconciliation fails, something has been double counted
and nothing downstream can be trusted.

In [6]:
cf = feat_july
assert cf["CompanyNumber"].is_unique, "CompanyNumber must be unique (one row per company)"
print("one row per company:", cf["CompanyNumber"].is_unique)

recon = int(cf.gaz_notice_count_total.sum()) == len(keyed)
print("notice counts reconcile:", recon,
      f"({int(cf.gaz_notice_count_total.sum()):,} vs {len(keyed):,} keyed notices)")
assert recon, "per-company totals must equal the keyed notice count"

distress_any = (cf.gaz_court_involved_flag | cf.gaz_has_liquidation |
                cf.gaz_has_administration).astype(bool)
print("companies with at least one formal distress signal:",
      f"{int(distress_any.sum()):,} ({distress_any.mean() * 100:.1f}%)")

print("\nseverity tier distribution:")
print(cf.gaz_severity_tier.value_counts().to_string())

print("\nmax distress stage distribution:")
print(cf.gaz_max_distress_stage.value_counts().sort_index().to_string())

print("\nmissing values (blanks expected for postcode and petition timing):")
miss = cf.isna().sum()
print(miss[miss > 0].to_string() if (miss > 0).any() else "none")

one row per company: True
notice counts reconcile: True (161,125 vs 161,125 keyed notices)
companies with at least one formal distress signal: 108,250 (99.0%)

severity tier distribution:
gaz_severity_tier
formal_insolvency    96580
terminal             10188
early_warning         2382
none                   183

max distress stage distribution:
gaz_max_distress_stage
0      183
1     2382
2    96580
3     9378
4      146
5      664

missing values (blanks expected for postcode and petition timing):
gaz_notice_postcode            51782
gaz_notice_postcode_area       51782
gaz_days_petition_to_order    100280


## Step 6: the four matching and provenance columns

NB10 added these in a later step, so the June file carries 56 columns rather than 52. They record
how each company was matched and which notices the row came from, which is what makes a row
auditable back to the original Gazette entry.

Every row here was matched on a company number printed in the notice text, so the match method is
constant. The columns still earn their place, because if name matching is ever added they become
the thing that separates a certain match from a probable one.

In [7]:
urls = (keyed.dropna(subset=["notice_url"])
             .groupby("CompanyNumber")["notice_url"]
             .apply(lambda s: " | ".join(s.astype(str).head(5))))

feat_july["gaz_has_company_number"] = 1
feat_july["gaz_match_method"] = "number"
feat_july["gaz_name_only_match_flag"] = 0
feat_july["gaz_source_notice_urls"] = (feat_july["CompanyNumber"].map(urls)
                                       .fillna(feat_july["gaz_latest_notice_url"]))

print("final shape:", feat_july.shape)
print("columns:", len(feat_july.columns))

final shape: (109333, 56)
columns: 56


## Step 7: save, without touching the June file

The rebuild goes to a new filename. The June file stays exactly as it is, so the comparison below
is against something still on disk rather than something we have already replaced. Nothing gets
overwritten until the numbers have been read and accepted.

In [8]:
feat_july.to_csv(FEATURES_JULY, index=False)
print("saved ->", FEATURES_JULY.name, f"({FEATURES_JULY.stat().st_size / 1e6:.1f} MB)")
print("June file untouched:", FEATURES_JUNE.exists(),
      f"({FEATURES_JUNE.stat().st_size / 1e6:.1f} MB)")

saved -> nb10_gazette_company_features_thru_2026-07.csv (34.6 MB)
June file untouched: True (33.7 MB)


## Step 8: what actually changed

Now the comparison. Three groups of companies matter here.

**New** companies appear in the rebuild but not in June, because their first ever notice landed in
July. **Common** companies appear in both, and are where the interesting movement is. There should
be no **dropped** companies at all, because notices are only ever added, never removed. If any show
up, something is wrong with the merge and the run should stop.

In [9]:
june = pd.read_csv(FEATURES_JUNE, dtype={"CompanyNumber": str})
june.columns = [c.strip() for c in june.columns]
june["CompanyNumber"] = june["CompanyNumber"].str.strip()

set_june = set(june["CompanyNumber"])
set_july = set(feat_july["CompanyNumber"])

new_cos = set_july - set_june
dropped = set_june - set_july
common = set_june & set_july

print(f"June feature file : {len(set_june):,} companies")
print(f"July rebuild      : {len(set_july):,} companies")
print(f"  new             : {len(new_cos):,}")
print(f"  common          : {len(common):,}")
print(f"  dropped         : {len(dropped):,}   (must be 0)")
assert not dropped, "companies should never disappear when notices are only added"

June feature file : 106,664 companies
July rebuild      : 109,333 companies
  new             : 2,669
  common          : 106,664
  dropped         : 0   (must be 0)


### Splitting the clock effect from the new-notice effect

This is the part that stops the comparison being misleading. The control table holds the notices
constant and only moves the reference date, so anything that differs between June and the control
is purely the clock. Anything that differs between the control and the rebuild is purely new
notices arriving.

The recency features are the ones exposed to the clock. A company whose last notice was 80 days
before 30 June is 111 days before 31 July, so its 90 day recency flag flips off without anything
happening to that company at all. That is correct behaviour, but it would be easy to misread as
distress fading, so it is worth quantifying rather than describing.

In [10]:
KEY_COLS = ["gaz_notice_count_total", "gaz_max_distress_stage", "gaz_severity_tier",
            "gaz_active_case_flag", "gaz_recent_notice_90d_flag", "gaz_recent_notice_365d_flag",
            "gaz_notice_count_12m", "gaz_days_since_latest_notice", "gaz_current_stage"]


def aligned(a, b, cols):
    'Line two feature tables up on the companies they share, as comparable strings.'
    idx = sorted(set(a["CompanyNumber"]) & set(b["CompanyNumber"]))
    A = a.set_index("CompanyNumber").reindex(idx)[cols].astype(str)
    B = b.set_index("CompanyNumber").reindex(idx)[cols].astype(str)
    return A, B, idx


def diff_report(a, b, cols, label_a, label_b):
    A, B, idx = aligned(a, b, cols)
    changed = (A != B)
    rows = changed.any(axis=1)
    print(f"{label_a} -> {label_b}: {int(rows.sum()):,} of {len(idx):,} shared companies changed "
          f"({rows.mean() * 100:.1f}%)")
    per_col = changed.sum().sort_values(ascending=False)
    print(per_col[per_col > 0].to_string() if (per_col > 0).any() else "  no column changed")
    return rows


print("=== clock only: same notices, reference date moved from 30 Jun to 31 Jul ===")
clock_rows = diff_report(june, feat_control, KEY_COLS, "June", "control")

print("\n=== new notices only: same reference date, July notices added ===")
notice_rows = diff_report(feat_control, feat_july, KEY_COLS, "control", "rebuild")

print("\n=== combined, which is what the June file to the rebuild actually looks like ===")
total_rows = diff_report(june, feat_july, KEY_COLS, "June", "rebuild")

=== clock only: same notices, reference date moved from 30 Jun to 31 Jul ===
June -> control: 106,664 of 106,664 shared companies changed (100.0%)
gaz_days_since_latest_notice    106664
gaz_recent_notice_90d_flag        3491
gaz_notice_count_12m              3354
gaz_recent_notice_365d_flag       2902
gaz_active_case_flag              2831

=== new notices only: same reference date, July notices added ===
control -> rebuild: 617 of 106,664 shared companies changed (0.6%)
gaz_notice_count_total          577
gaz_days_since_latest_notice    577
gaz_notice_count_12m            577
gaz_current_stage               518
gaz_active_case_flag            299
gaz_max_distress_stage          257
gaz_severity_tier               248
gaz_recent_notice_90d_flag      175
gaz_recent_notice_365d_flag      58

=== combined, which is what the June file to the rebuild actually looks like ===
June -> rebuild: 106,664 of 106,664 shared companies changed (100.0%)
gaz_days_since_latest_notice    106659
gaz_notic

June -> control: 106,664 of 106,664 shared companies changed (100.0%)
gaz_days_since_latest_notice    106664
gaz_recent_notice_90d_flag        3491
gaz_notice_count_12m              3354
gaz_recent_notice_365d_flag       2902
gaz_active_case_flag              2831

=== new notices only: same reference date, July notices added ===


control -> rebuild: 617 of 106,664 shared companies changed (0.6%)
gaz_notice_count_total          577
gaz_days_since_latest_notice    577
gaz_notice_count_12m            577
gaz_current_stage               518
gaz_active_case_flag            299
gaz_max_distress_stage          257
gaz_severity_tier               248
gaz_recent_notice_90d_flag      175
gaz_recent_notice_365d_flag      58

=== combined, which is what the June file to the rebuild actually looks like ===


June -> rebuild: 106,664 of 106,664 shared companies changed (100.0%)
gaz_days_since_latest_notice    106659
gaz_notice_count_12m              3920
gaz_recent_notice_90d_flag        3574
gaz_active_case_flag              3122
gaz_recent_notice_365d_flag       2950
gaz_notice_count_total             577
gaz_current_stage                  518
gaz_max_distress_stage             257
gaz_severity_tier                  248


### Movement on the distress ladder

The question the business side will ask is whether companies got worse. The ladder makes that
answerable, since `gaz_max_distress_stage` only ever moves up as new notices arrive. A company
cannot climb down it, because the feature is a maximum over everything ever seen.

So any movement here is real deterioration, not noise, and the crosstab below shows exactly which
step each company moved from and to.

In [11]:
A, B, idx = aligned(june, feat_july, ["gaz_max_distress_stage"])
old_stage = pd.to_numeric(A["gaz_max_distress_stage"], errors="coerce")
new_stage = pd.to_numeric(B["gaz_max_distress_stage"], errors="coerce")

moved = old_stage != new_stage
print(f"companies whose max distress stage moved: {int(moved.sum()):,}")
print(f"  worsened (moved up)  : {int((new_stage > old_stage).sum()):,}")
print(f"  improved (moved down): {int((new_stage < old_stage).sum()):,}   (expected 0)")

if moved.any():
    STAGE_LABEL = {0: "0 none", 1: "1 petition", 2: "2 formal insolvency",
                   3: "3 creditor process", 4: "4 dividend", 5: "5 closing"}
    ct = pd.crosstab(old_stage[moved].map(STAGE_LABEL), new_stage[moved].map(STAGE_LABEL))
    print("\nfrom (rows) -> to (columns):")
    print(ct.to_string())

print("\nseverity tier, June against the rebuild, shared companies only:")
tier = pd.DataFrame({
    "june": june.set_index("CompanyNumber").reindex(idx)["gaz_severity_tier"],
    "july": feat_july.set_index("CompanyNumber").reindex(idx)["gaz_severity_tier"],
})
print(pd.crosstab(tier["june"], tier["july"]).to_string())

companies whose max distress stage moved: 257
  worsened (moved up)  : 257
  improved (moved down): 0   (expected 0)

from (rows) -> to (columns):
gaz_max_distress_stage  1 petition  2 formal insolvency  3 creditor process  4 dividend  5 closing
gaz_max_distress_stage                                                                            
0 none                           1                    1                   1           0          0
1 petition                       0                  215                  14           0          0
2 formal insolvency              0                    0                  10           4          2
3 creditor process               0                    0                   0           0          9

severity tier, June against the rebuild, shared companies only:
july               early_warning  formal_insolvency  none  terminal
june                                                               
early_warning               2053                215     0 

july               early_warning  formal_insolvency  none  terminal
june                                                               
early_warning               2053                215     0        14
formal_insolvency              0              94397     0        16
none                           1                  1   176         1
terminal                       0                  0     0      9790


## Step 9: the 1,033 reconciliation

NB10 reported 1,033 companies on the watchlist. That number came from joining the Gazette features
against `filtered_bb_sme_sectors.csv`, the original universe of 1.37M companies filtered to Active
status only. So 1,033 was never the number of distressed companies. It was the number of companies
that were **still trading** and carrying a Gazette signal, which is a much narrower and much more
interesting question.

Once inactive companies are included, the picture changes completely, and it should. A company in
liquidation is not Active, so the Active-only filter was systematically removing exactly the
companies most likely to have insolvency notices against them.

The cell below joins against the widened universe of 1.49M companies across all statuses and breaks
the result down by lifecycle, which is where the reconciliation becomes obvious.

In [12]:
UNIVERSE = DATA / "filtered_bb_sme_sectors_all_status.csv"

print("reading the widened universe (668 MB, this takes a moment) ...")
uni = pd.read_csv(
    UNIVERSE, dtype=str,
    usecols=lambda c: c.strip() in {"CompanyName", "CompanyNumber", "CompanyStatus",
                                    "sector", "segment", "lifecycle"},
)
uni.columns = [c.strip() for c in uni.columns]
uni["CompanyNumber"] = uni["CompanyNumber"].str.strip()
uni = uni.drop_duplicates("CompanyNumber")
print(f"universe: {len(uni):,} companies")
print(uni["lifecycle"].value_counts().to_string())

reading the widened universe (668 MB, this takes a moment) ...
universe: 1,493,972 companies
lifecycle
Trading       1372321
Fading          97652
Insolvent       23828
Distressed        171


universe: 1,493,972 companies
lifecycle
Trading       1372321
Fading          97652
Insolvent       23828
Distressed        171


In [13]:
merged = uni.merge(feat_july.drop(columns=["company_name"]), on="CompanyNumber", how="left")
merged["gaz_has_any_notice"] = merged["gaz_has_any_notice"].fillna(0).astype(int)

flagged = merged[merged["gaz_has_any_notice"] == 1]
active_flagged = flagged[flagged["CompanyStatus"].str.strip() == "Active"]

print(f"companies in the widened universe with a Gazette signal: {len(flagged):,} "
      f"({len(flagged) / len(merged) * 100:.2f}%)")
print(f"  of which still Active: {len(active_flagged):,}   <- comparable to NB10's 1,033")
print(f"  inactive             : {len(flagged) - len(active_flagged):,}   "
      f"<- invisible to the Active-only join")

print("\nflagged companies by lifecycle:")
print(flagged["lifecycle"].value_counts().to_string())

print("\nflagged Active companies by severity tier:")
print(active_flagged["gaz_severity_tier"].value_counts().to_string())

companies in the widened universe with a Gazette signal: 19,814 (1.33%)
  of which still Active: 1,659   <- comparable to NB10's 1,033
  inactive             : 18,155   <- invisible to the Active-only join

flagged companies by lifecycle:
lifecycle
Insolvent     17881
Trading        1659
Fading          244
Distressed       30

flagged Active companies by severity tier:
gaz_severity_tier
formal_insolvency    1101
early_warning         321
terminal              216
none                   21


## Step 10: the refreshed watchlist

The watchlist is the practical output, the companies a relationship manager would actually look at.
It is written to a new filename alongside the June one, so both are available while the comparison
is being reviewed.

Worth being clear about what this version is and is not. It is built on the widened universe, which
is still derived from the June Companies House snapshot, so company status here is as at 1 June even
though the notices now run to 31 July. That gap closes at step 4 of the plan, when the universe is
rebuilt on the August snapshot. Until then the notice side is current and the company side is not,
and any reading of this file should carry that caveat.

In [14]:
watch = flagged.copy()
watch.to_csv(WATCHLIST_JULY, index=False)

num = lambda col: int(pd.to_numeric(watch[col], errors="coerce").fillna(0).sum())
print(f"watchlist rows: {len(watch):,}")
print("  recent notice (90d):", f"{num('gaz_recent_notice_90d_flag'):,}")
print("  open case          :", f"{num('gaz_active_case_flag'):,}")
print("  court involved     :", f"{num('gaz_court_involved_flag'):,}")

print("\nby sector:")
print(watch["sector"].value_counts().to_string())

print("\nsaved ->", WATCHLIST_JULY.name)

watchlist rows: 19,814
  recent notice (90d): 2,161
  open case          : 9,409
  court involved     : 3,354

by sector:
sector
Technology, legal & professional    10246
Manufacturing                        5295
Fast growth & emerging               4273

saved -> nb10_active_gazette_watchlist_thru_2026-07.csv
